# Capstone G3: Head trauma / stroke on brain CT -- and who's missing from the data

Brain CT scans, labeled normal vs stroke (a bleed or blockage -- what an ER checks a head-injury patient for). You'll build the classifier AND uncover a quieter problem: who is *not* in this dataset.

**Your group's priority: fairness you can't even measure.** This dataset gives you an image and a label and NOTHING else -- no age, no sex, no scanner, no hospital. So you literally *can't* check whether it works equally for everyone. That blank is your finding: a dataset that hides who's in it.

**Your goal (the rubric):** build it, check honestly how well it works, find a case where it fails, defend one choice you made, and make sure *both* partners can explain everything.

### How to use this notebook (read this first)

- A notebook is a stack of **cells**. A gray cell is **code**; this white cell is a note.
- To run a code cell, click it and press **Shift+Enter**. Run them **top to bottom, in order** --
  later cells use things earlier cells made.
- **Red text is normal**, not a disaster -- it's usually a warning. A real error stops with a
  message; read the last line, or paste it to Claude and ask "what does this mean?"
- You will not understand every line, and that's fine. But you should be able to say, in plain
  English, *what each cell is for*. Stuck on a line? Ask Claude: **"explain this cell line by line."**
- Nothing you click here can break anything. Experiment.

### You have Claude Code -- so here's your *real* job

You can already make a computer write code: just describe what you want to Claude Code and it will
build it. So **writing code is no longer the hard part.** The skill that actually makes you good at
this -- the thing this notebook is training -- is **judgment**:

1. **What are you trying to build?** "Accurate" isn't a goal. A cancer screen that must never miss a
   tumor is a *different tool* than one that's just right-on-average. Decide what "good" means first.
2. **What tools exist?** Different models, different ways to prep the data, different ways to grade a
   model. This notebook is a guided tour of that toolbox.
3. **Why does each tool exist** -- what problem it was invented to solve.
4. **How do you pick the right one?** That's the whole game. The wrong tool gives a wrong answer even
   with flawless code.

So: read the notes, play with the dials, and when you want to change or add something, **describe it
to Claude Code and let it write the code** -- then be ready to explain *why you chose it*. That "why"
is what you're graded on, and it's what a real scientist or doctor would ask you.

In [ ]:
# WHAT THIS DOES: gets the course files onto this Colab machine and installs the tools we need.
# (On your own computer it does nothing.) Just run it once and wait for "setup ok".
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g3_brain_ct")
sys.path.insert(0, "../..")            # so we can import the course helper files
sys.path.insert(0, "../../../_shared")
import colab_setup
colab_setup.ensure(*colab_setup.DAY1)   # our own dataset -> no medmnist needed

### First, some plain-English vocabulary

- A **model** is a rule the computer writes *by itself* after seeing lots of examples -- like a kid
  who learns "cat vs dog" from photos, not from a dictionary definition.
- **Training** = showing it labeled examples so it can find that rule.
- To a computer, an **image is just a grid of numbers** (how bright each pixel is). The model does
  math on those numbers. It never "sees" a picture the way you do.
- We split our data into a **training set** (the model studies these) and a **test set** (held back,
  like a real exam) -- because a model that just *memorized* the study set would ace the homework and
  flunk the exam. The **test score is the only one that counts.**

In [ ]:
# WHAT THIS DOES: loads the images, splits them into train/test, and tells us the class names.
import torch, sys
sys.path.insert(0, "../..")
import capstone_common as cc

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)   # 'cuda' = a GPU is helping (fast). 'cpu' = slower but still fine.
# 'brainct' is a small brain-CT dataset we prepared for you. It loads just like any other.

FLAG = "brainct"
train_loader, val_loader, test_loader, n_classes, task = cc.get_loaders(FLAG, size=64)
CLASS_NAMES = cc.class_names(FLAG)
print("classes:", n_classes, "->", CLASS_NAMES, "| task:", task)

### Look at the data first
**What this does:** draws a few images with their labels. Always eyeball your data before modeling -- can *you* even tell the classes apart?

In [ ]:
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_loader))   # grab one batch of images
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, im, lb in zip(axes, imgs, labels):
    ax.imshow(im[0], cmap="gray")
    ax.set_title(CLASS_NAMES[int(lb)], fontsize=8); ax.axis("off")
plt.tight_layout()

### Build a first model (the baseline)

**What this does:** trains a model in a few seconds using a shortcut called **transfer learning**,
then prints its **test accuracy** (the fraction it got right on images it never studied).

**Why this works -- the analogy:** instead of teaching a computer to see *from scratch* (which needs
millions of images), we **borrow a network that already learned to see** from millions of everyday
photos. It already knows edges, shapes, and textures. We keep all of that ("**freeze**" it) and only
train a tiny final decision layer (a new "**head**") for our specific job. It's like hiring someone
who already speaks the language and just teaching them the medical words.

In [ ]:
model = cc.make_baseline(n_classes)                                   # borrowed "eyes" + a fresh decision layer
model = cc.train(model, train_loader, val_loader, epochs=3, device=device)  # 3 passes over the data
test_acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
print(f"\nbaseline TEST accuracy: {test_acc:.3f}")   # e.g. 0.800 = right 80% of the time

### Now build your OWN model (interactive -- no coding needed)

**What this does:** gives you dials. **Pick options, press "Run Interact", read the test accuracy.**
Change **one dial at a time** so you can tell what actually helped. Write every result down --
that log is half your presentation.

**What each dial means, in plain terms:**
- **backbone** -- which "brain" to borrow. `resnet18` is small and fast; the others are bigger
  (sometimes smarter, sometimes just slower).
- **pretrained** -- ON = start from the brain that already learned to see (usually much better).
  OFF = start from a *blank* brain. With only a few thousand images, blank is like learning to read
  and diagnose at the same time -- too much at once, so it does worse. **Try toggling this; it's the
  single biggest lesson here.**
- **unfreeze** -- also re-train the borrowed brain, not just the head. More power, but easier to
  **overfit** (memorize instead of learn).
- **augment** -- show each image flipped/rotated. Teaches "the lesion," not "the lesion is top-left."
- **epochs** -- how many times it studies the whole set. More isn't always better (it can start memorizing).

**How to choose (a real strategy, not random clicking):** change **pretrained** first -- it usually
matters most. Then try **augment**, then a bigger **backbone**. **One dial at a time**, or you won't
know which change helped. And prefer the *simplest* setup that hits your goal: a smaller model you
understand beats a bigger one you can't explain.

In [ ]:
from ipywidgets import interact_manual, Dropdown, Checkbox, IntSlider

def build_model(backbone="resnet18", pretrained=True, unfreeze_backbone=False, augment=False, epochs=3):
    global model, train_loader, val_loader, test_loader
    train_loader, val_loader, test_loader, n_cls, _ = cc.get_loaders(FLAG, size=64, augment=augment)
    model = cc.make_model(n_cls, backbone=backbone, pretrained=pretrained, unfreeze_backbone=unfreeze_backbone)
    model = cc.train(model, train_loader, val_loader,
                     epochs=epochs, lr=(1e-4 if unfreeze_backbone else 1e-3), device=device)
    acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
    print(f"\n>>> TEST accuracy = {acc:.3f}   "
          f"[{backbone}, pretrained={pretrained}, unfreeze={unfreeze_backbone}, augment={augment}, {epochs}ep]")

try:
    interact_manual(build_model,
        backbone=Dropdown(options=cc.BACKBONES, value="resnet18", description="backbone"),
        pretrained=Checkbox(value=True, description="pretrained"),
        unfreeze_backbone=Checkbox(value=False, description="unfreeze"),
        augment=Checkbox(value=False, description="augment"),
        epochs=IntSlider(value=3, min=1, max=10, description="epochs"))
except ImportError:
    build_model()

### Grade your model like a regulator

A high accuracy is **not** enough for medicine. Yesterday you set the *rules*; now hold **your**
model to them. Pick a tool below and run it. Your group's priority is **fairness you can't even measure**, so the
menu starts on **Failure analysis** -- but try the others too.

**What each tool tells you (in plain terms):**
- **Failure analysis** -- which scans does it miss? Any visible pattern?
- **Fairness** -- here it can only compare *classes* (normal vs stroke), because the data records no age/sex/race at all. Notice what you're *unable* to check -- that's the point.
- **Grad-CAM** -- is it looking at brain tissue, or cheating off the skull / scanner edges?

**How to choose which audit:** start from *what would hurt a patient most* in your job. Screening
where a miss is deadly? Lead with **failure analysis**. Worried the model treats groups unequally?
**Fairness**. Not sure it's even looking at the right thing? **Grad-CAM**. The priority drives the tool.

*(Train a model first -- run the baseline or the builder above.)*

In [ ]:
from ipywidgets import interact_manual, Dropdown

TOOLS = list(cc.REGULATOR_TOOLS)
DEFAULT = [t for t in TOOLS if t.startswith("Failure")][0]

def audit(tool):
    if "model" not in globals():
        print("Train a model first (run the baseline or the builder above)."); return
    cc.REGULATOR_TOOLS[tool](model, test_loader, device=device, class_names=CLASS_NAMES)

try:
    interact_manual(audit, tool=Dropdown(options=TOOLS, value=DEFAULT,
                                          description="priority", style={"description_width": "initial"}))
except ImportError:
    audit(DEFAULT)

---
## Part 2 -- the fairness the brain CT can't give you

Your CT model works, but it records **nothing about the patient** -- no age, no sex, no race. So you
**cannot check** whether it misses strokes more often in one group than another. That blank is the
whole problem.

The fix: bring in a **different** stroke dataset -- rows of numbers per patient (age, sex, blood
pressure, glucose, smoking) that **does** record who each person is. Now you can build a stroke-risk
model *and* audit it. This is the same idea as Day 2's tabular work.

In [ ]:
# Install the tabular tools + load the stroke dataset (it ships with the course repo).
colab_setup.ensure("catboost", "tabpfn==2.2.1", "scikit-learn", "pandas")
import capstone_tabular as ct                      # the tabular helpers (like Day 2)
df, meta = ct.load_stroke()                        # 5,110 patients, each a row of numbers
print(meta["name"], "->", df.shape[0], "patients")
print("features:", meta["features"])
print("we can audit by:", meta["group"], meta["group_names"])
df.head()

**A trap to notice first:** only about **5%** of these patients had a stroke. So a lazy model that
says "no stroke" for *everyone* is right 95% of the time and completely useless. Accuracy is the wrong
score here -- watch **AUC** (0.5 = coin flip, 1.0 = perfect) and, most of all, **how many real strokes
you actually catch** in each group.

In [ ]:
# Build a stroke-risk model: pick features + a model, hit Run, read the score.
from ipywidgets import interact_manual, SelectMultiple, Dropdown

def build(features, model):
    if not features:
        print("Pick at least one feature."); return
    ct.fit_eval(df, list(features), meta["target"], model=model)

try:
    interact_manual(build,
        features=SelectMultiple(options=meta["features"], value=tuple(meta["features"]),
                                description="features", rows=10, style={"description_width": "initial"}),
        model=Dropdown(options=list(ct.make_models()), value="Random Forest", description="model"))
except ImportError:
    ct.fit_eval(df, meta["features"], meta["target"], model="Random Forest")

### Audit by sex -- the check the CT could not do

Train once, then measure how well the model does **separately for women and men**. This is exactly the
fairness question the brain-CT dataset made impossible -- and here, because sex is recorded, you can
answer it. If the model catches strokes less often in one group, that's a real problem to fix
(e.g. a group-aware threshold).

In [ ]:
from ipywidgets import interact_manual, Dropdown

def fairness(model):
    ct.audit_by_group(df, meta["features"], meta["target"],
                      meta["group"], meta.get("group_names"), model=model)

try:
    interact_manual(fairness,
        model=Dropdown(options=list(ct.make_models()), value="Random Forest", description="model"))
except ImportError:
    ct.audit_by_group(df, meta["features"], meta["target"], meta["group"], meta.get("group_names"))

### One strong approach (peek only if you're stuck)

*This is roughly what the worked solution does. It's a nudge, not the code -- describe these steps to
Claude and build them yourself, then be ready to explain why.*

Build the stroke detector first (transfer learning on the CT slices). Then notice the honest catch: this image dataset records **no demographics** (age/sex/race), so you literally **cannot audit** whether it is fair. The strong move is to bring in a separate **tabular stroke dataset that DOES record sex and age**, build a stroke-risk model there, and check whether it catches strokes as often in women as in men (equalize with group-aware thresholds if not). The lesson: **recording demographics is what makes fairness checkable** -- the imaging set hides who is in it.

### Where to take it (ask Claude to help)
- List every fact a hospital would need before trusting this (age, sex, race, scanner, site) and which ones this dataset gives you. The gap is your headline.
- Use **Monitoring** to add noise -- a real ER scanner is messier than clean training data.
- Try to find a brain-CT dataset that *does* report demographics. How rare is that?